This notebook is dedicated for building the tensorflow model innitizer and experimentation with it. 

In [1]:
import tensorflow as tf

c:\Users\puket\myenv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
from  model_configs.config_reader import config

cfg=config("lobotomy_shakespeare.yaml")
model =cfg.model

In [ ]:
from Attention_Mechanisms.core.transformer_math import dot_product_attention as attention
from Attention_Mechanisms.backends.tf_backend import tf_backend
from functools import partial

class n_Vaswani_Layer(tf.keras.layers.Layer):

    def __init__(self, *,num_heads,num_outputs, activity_regularizer = None, trainable = True, dtype = None, autocast = True, name = None, input_dim = None, input_shape = None):
        self.bknd = tf_backend(False)
        self.num_heads = num_heads
        super().__init__(activity_regularizer=activity_regularizer, trainable=trainable, dtype=dtype, autocast=autocast, name=name, input_dim=input_dim, input_shape=input_shape)
        self.W_O = tf.keras.layers.Dense(num_outputs,dtype=tf.float32)
        self.head = partial(attention,backend=self.bknd)
        self.head_dim = num_outputs//num_heads
        self.W_q= []
        self.W_k= []
        self.W_v= []
        for __ in range(self.num_heads):
            self.W_q.append(tf.keras.layers.Dense(self.head_dim,dtype=tf.float32))
            self.W_k.append(tf.keras.layers.Dense(self.head_dim,dtype=tf.float32))
            self.W_v.append(tf.keras.layers.Dense(self.head_dim,dtype=tf.float32))



    
    def call(self,v):
        heads = []
        for index in range(self.num_heads):
            Q = self.W_q[index](v)
            K = self.W_k[index](v)
            V = self.W_v[index](v)

            heads.append(self.head(Q,K,V))


        attn_out = tf.concat(heads,axis=-1)
        output = self.W_O(attn_out)
        return output
   



In [9]:
#Check

Layer = n_Vaswani_Layer(num_heads=model.attention_heads,num_outputs=model.context_length)

import numpy as np

X = np.random.standard_normal((model.embedding_dim,model.context_length))
X =tf.cast(X,tf.float32)

Layer(X)

<tf.Tensor: shape=(100, 200), dtype=float32, numpy=
array([[-0.00263336, -0.25753143, -0.1511325 , ..., -0.18082671,
         0.27490842, -0.05691484],
       [ 0.10424925,  0.08310428,  0.04430081, ...,  0.0586725 ,
        -0.35805774, -0.05979346],
       [ 0.06526364,  0.06865274,  0.02416215, ...,  0.41539264,
         0.27552086,  0.07573871],
       ...,
       [ 0.9648355 ,  0.08345819,  0.14844659, ...,  0.23459423,
         0.24764048, -0.19500701],
       [ 0.01202284, -0.04389996,  0.23216727, ...,  0.06747375,
        -0.24227294,  0.33413285],
       [ 0.3338499 , -0.52503675, -0.0468441 , ..., -0.34681097,
        -0.20861468, -0.16354033]], shape=(100, 200), dtype=float32)>

In [26]:
# Functional API model construction

input = tf.keras.Input(shape = (model.context_length,))

#Embedder: 

Lookup = tf.keras.layers.Embedding(input_dim= model.context_length, output_dim=model.context_length)(input)
Squeezed = tf.keras.layers.Lambda(lambda x: tf.squeeze(x, axis=0))(Lookup)


# Attention layers:
for __ in range(model.layers):
    Transfrmd=n_Vaswani_Layer(num_heads=model.attention_heads,num_outputs=model.context_length)(Squeezed)

    Skip = tf.keras.layers.Add()([Transfrmd, Lookup])

    Normed = tf.keras.layers.LayerNormalization()(Skip)

    FF1=tf.keras.layers.Dense(model.context_length,activation=tf.keras.activations.gelu)(Normed)
    FF2 = tf.keras.layers.Dense(model.context_length)(FF1)

    Skip2 = tf.keras.layers.Add()([Normed,FF2])

    Normed2 = tf.keras.layers.LayerNormalization()(Skip2)


prediction = tf.keras.layers.Dense(model.vocab_size)(Normed2)
GPT = tf.keras.Model(inputs =input, outputs= prediction)


In [27]:
GPT.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_14      │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_13        │ (None, 200, 200)  │     40,000 │ input_layer_14[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_8 (Lambda)   │ (200, 200)        │          0 │ embedding_13[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ n__vaswani__layer_… │ (200, 200)        │    160,800 │ lambda_8[0][0]    │
│ (n_Vaswani_Layer)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_14 (Add)        │ (200, 200, 200)   │          0 │ n__vaswani__laye… │
│                     │                   │            │ embedding_13[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (200, 200, 200)   │        400 │ add_14[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_127 (Dense)   │ (200, 200, 200)   │     40,200 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_128 (Dense)   │ (200, 200, 200)   │     40,200 │ dense_127[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_15 (Add)        │ (200, 200, 200)   │          0 │ layer_normalizat… │
│                     │                   │            │ dense_128[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (200, 200, 200)   │        400 │ add_15[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_129 (Dense)   │ (200, 200, 10000) │  2,010,000 │ layer_normalizat… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,292,000 (8.74 MB)

 Trainable params: 2,292,000 (8.74 MB)

 Non-trainable params: 0 (0.00 B)